# Reading

In [164]:
import pandas as pd
import os

from pandas.core.indexes import category

In [165]:
parent_path = "C:/Users/quintyn_come/Codex/AIBT102/"

In [166]:
commerces_paris = pd.read_csv(os.path.join(parent_path, "scraping_and_preprocessing/processed_data/commerces/commerces_osm_paris.csv"))
commerces_haute_garonne = pd.read_csv(os.path.join(parent_path, "scraping_and_preprocessing/processed_data/commerces/commerces_osm_Haute-Garonne.csv"))

In [167]:
list_of_all_dfs = [
    commerces_paris,
    commerces_haute_garonne,
]

In [168]:
businesses = pd.concat(list_of_all_dfs, ignore_index=True)

In [169]:
businesses.head(5)

,source,name,activity_query,category,category_type,address,lat,lon,opening_hours,phone,website,siret,osm_type,osm_id,osm_url,iris_code,iris,iris_name,commune_name,iris_type
0,OpenStreetMap,Relais Total Du Pont Aval,all_commercial_establishments_paris,convenience,shop,75015,48.834677,2.266146,24/7,+33 1 53 78 05 01,https://store.totalenergies.fr/en_EN/NF078037,NaN,node,25213384,https://www.openstreetmap.org/node/25213384,751156001,6001,Javel 1,Paris 15e Arrondissement,H
1,OpenStreetMap,Relais des Chaufourniers,all_commercial_establishments_paris,convenience;gas,shop,75019,48.880181,2.370744,24/7,NaN,NaN,NaN,node,25214653,https://www.openstreetmap.org/node/25214653,751197301,7301,Villette 1,Paris 19e Arrondissement,H
2,OpenStreetMap,Total,all_commercial_establishments_paris,convenience;gas,shop,NaN,48.831580,2.315822,24/7,NaN,NaN,NaN,node,66492656,https://www.openstreetmap.org/node/66492656,751145616,5616,Plaisance 16,Paris 14e Arrondissement,H
3,OpenStreetMap,Haltes-garderie municipale,all_commercial_establishments_paris,kindergarten,amenity,NaN,48.831463,2.316597,NaN,NaN,https://mairie14.paris.fr/pages/creches-du-14e...,NaN,node,158268042,https://www.openstreetmap.org/node/158268042,751145616,5616,Plaisance 16,Paris 14e Arrondissement,H
4,OpenStreetMap,Moulin Créole,all_commercial_establishments_paris,restaurant,amenity,NaN,48.829834,2.322899,07:00-23:30,+33 1 45 39 31 31,https://www.moulin-creole-paris.fr,NaN,node,158603712,https://www.openstreetmap.org/node/158603712,751145613,5613,Plaisance 13,Paris 14e Arrondissement,H


Let's drop the columns that aren't relevant for our project:

In [170]:
businesses.drop(columns=["source", "activity_query", "address", "website", "siret", "osm_type", "osm_url", "iris", "iris_name", "commune_name", "iris_type", "phone"], inplace=True)

C:\Users\quintyn_come\AppData\Local\Temp\ipykernel_37736\971150341.py:1: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  businesses.drop(columns=["source", "activity_query", "address", "website", "siret", "osm_type", "osm_url", "iris", "iris_name", "commune_name", "iris_type", "phone"], inplace=True)


# Cleaning

Let's check the number of NaN values:

In [171]:
businesses.isna().sum()

name              4534
category             0
category_type        0
lat                  0
lon                  0
opening_hours    40168
osm_id               0
iris_code            0
dtype: int64

In [172]:
print('Number of rows:', businesses.shape[0])

Number of rows: 74412


It seems that more that half opening hours are missing, we can't really replace them so we will drop them but for a better study it would be interesting to keep them.
Also some shops don't have a name I will drop those rows.

In [173]:
businesses.drop(columns=["opening_hours"], inplace=True)
businesses = businesses.dropna(subset=["name"])

C:\Users\quintyn_come\AppData\Local\Temp\ipykernel_37736\2766881136.py:1: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  businesses.drop(columns=["opening_hours"], inplace=True)


Let's check the number of NaN now:

In [174]:
businesses.isna().sum()

name             0
category         0
category_type    0
lat              0
lon              0
osm_id           0
iris_code        0
dtype: int64

In [175]:
print('Number of rows:', businesses.shape[0])

Number of rows: 69878


In [176]:
businesses = businesses.rename(columns={
    'iris_code': 'IRIS',
})

Great there aren't any NaN anymore.

# Category

In [177]:
businesses.head(5)

,name,category,category_type,lat,lon,osm_id,IRIS
0,Relais Total Du Pont Aval,convenience,shop,48.834677,2.266146,25213384,751156001
1,Relais des Chaufourniers,convenience;gas,shop,48.880181,2.370744,25214653,751197301
2,Total,convenience;gas,shop,48.831580,2.315822,66492656,751145616
3,Haltes-garderie municipale,kindergarten,amenity,48.831463,2.316597,158268042,751145616
4,Moulin Créole,restaurant,amenity,48.829834,2.322899,158603712,751145613


Let's check the number of categories and category types in our project:

In [178]:
print("Number of different categories:", len(businesses["category"].unique()))

Number of different categories: 516


It seems that there are many categories ant that a business can have multpiple categories. Let's create a csv for the categories and a csv for the businesses.

The table to link the 0:n relationsip (this one is temporary just the time to create the categories table)

In [179]:
link = (
    businesses[["osm_id", "category", "category_type"]]
    .assign(category=lambda x: x["category"].str.split(";"))
    .explode("category")
)
link["category"] = link["category"].str.strip()
link = link.merge(
    businesses[["osm_id"]],
    on="osm_id",
)

In [180]:
link.head()

,osm_id,category,category_type
0,25213384,convenience,shop
1,25214653,convenience,shop
2,25214653,gas,shop
3,66492656,convenience,shop
4,66492656,gas,shop


The categories table

In [181]:
categories = (
    link[["category", "category_type"]]
    .dropna(subset=["category"])
    .drop_duplicates()
    .sort_values(["category", "category_type"])
    .reset_index(drop=True)
)

categories.insert(
    0,
    "category_id",
    range(1, len(categories) + 1)
)

In [182]:
categories.head()

,category_id,category,category_type
0,1,Aide à la personne,office
1,2,"Climatisation, énergies renouvelables",craft
2,3,accessories,shop
3,4,accountant,office
4,5,administration,office


The real link table:

In [183]:
link = (
    link
    .merge(categories, on="category", how="left")
    [["osm_id", "category_id"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

In [184]:
link.head()

,osm_id,category_id
0,25213384,127
1,25214653,127
2,25214653,214
3,66492656,127
4,66492656,214


Let's drop the category and category_type from the original business table:

In [185]:
businesses = businesses.drop(['category', 'category_type'], axis=1)

In [186]:
businesses.head()

,name,lat,lon,osm_id,IRIS
0,Relais Total Du Pont Aval,48.834677,2.266146,25213384,751156001
1,Relais des Chaufourniers,48.880181,2.370744,25214653,751197301
2,Total,48.831580,2.315822,66492656,751145616
3,Haltes-garderie municipale,48.831463,2.316597,158268042,751145616
4,Moulin Créole,48.829834,2.322899,158603712,751145613


# Saving

In [187]:
output_dir = os.path.join(parent_path, "cleaning/cleaned_data")
output_name = "businesses.csv"
output_path = os.path.join(output_dir, output_name)
os.makedirs(output_dir, exist_ok=True)
businesses.to_csv(
    output_path,
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

In [188]:
output_dir = os.path.join(parent_path, "cleaning/cleaned_data")
output_name = "categories.csv"
output_path = os.path.join(output_dir, output_name)
os.makedirs(output_dir, exist_ok=True)
categories.to_csv(
    output_path,
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

In [189]:
output_dir = os.path.join(parent_path, "cleaning/cleaned_data")
output_name = "category_business_link.csv"
output_path = os.path.join(output_dir, output_name)
os.makedirs(output_dir, exist_ok=True)
link.to_csv(
    output_path,
    sep=";",
    index=False,
    encoding="utf-8-sig"
)